<a href="https://colab.research.google.com/github/dataguirre/curso-ia-ciencia-de-datos/blob/main/workshops/02-workshop-uso-llms-solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IA para ciencia de datos: Workshop 2 (soluciones)

En este workshop exploramos las distintas formas en que se puede **usar** un LLM que ya esta entrenado:

1. A traves de la **API de un tercero** (Groq).
2. Levantando tu **propio servidor** (Ollama).
3. A traves de la **libreria `transformers`**, cargando el modelo directamente en memoria.

Despues veremos que pasa cuando un modelo **no cabe** en nuestros recursos computacionales, y como los parametros de generacion (temperatura, top-p, etc.) cambian la respuesta.

> **Antes de empezar:** ve a `Entorno de ejecucion > Cambiar tipo de entorno de ejecucion` y selecciona **GPU (T4)**. Las Actividades 3 y 4 la necesitan.


## Actividad 1: Via API de terceros (Groq)

### Objetivo

Implementar la funcion `preguntar_groq(prompt, system_prompt=None, modelo=MODELO_GROQ, temperature=0.7)`, que envia un mensaje a un LLM alojado en Groq y devuelve el texto de su respuesta.

Esto replica el patron cliente-servidor que vimos en clase: tu notebook es el **cliente**, Groq es el **servidor** que tiene el modelo.

### Antes de empezar: obten tu API key

1. Crea una cuenta gratuita en https://console.groq.com
2. Genera una API key en la seccion *API Keys*.
3. En Colab, abre el icono de llave (Secrets) en el panel izquierdo, agrega un secreto llamado `GROQ_API_KEY` con tu llave y activalo para este notebook.


In [ ]:
!pip install -q groq

from groq import Groq
from google.colab import userdata

# Si Groq llega a deprecar este modelo, revisa los modelos vigentes en
# https://console.groq.com/docs/models y cambia este valor.
MODELO_GROQ = "openai/gpt-oss-20b"

client = Groq(api_key=userdata.get("GROQ_API_KEY"))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 5.4 MB/s eta 0:00:00


#### Tarea 1: Construir la funcion `preguntar_groq`

### Requisitos

1. Si `system_prompt` no es `None`, el primer mensaje debe ser `{"role": "system", "content": system_prompt}`.
2. Despues (o al inicio, si no hay system prompt), agregar `{"role": "user", "content": prompt}`.
3. Devolver el **texto** de la respuesta (un string), no el objeto completo.


In [ ]:
def preguntar_groq(prompt: str, system_prompt: str = None, modelo: str = MODELO_GROQ, temperature: float = 0.7) -> str:
  """Envia un prompt (y opcionalmente un system prompt) a un LLM de Groq y devuelve el texto de la respuesta."""

  mensajes = []

  if system_prompt:
    mensajes.append({"role": "system", "content": system_prompt})

  mensajes.append({"role": "user", "content": prompt})

  respuesta = client.chat.completions.create(
      model=modelo,
      messages=mensajes,
      temperature=temperature,
  )

  return respuesta.choices[0].message.content


**Evaluacion de implementacion**

In [ ]:
# @title
# Celda de validacion. No modificar.
from unittest.mock import patch, MagicMock

def _respuesta_groq_falsa(texto="Respuesta simulada"):
  resp = MagicMock()
  resp.choices = [MagicMock()]
  resp.choices[0].message.content = texto
  return resp

def _validar_preguntar_groq():
  if "preguntar_groq" not in globals():
    print("\u2717 Todavia no existe 'preguntar_groq'. Ejecuta la celda anterior.")
    return

  fallas = []

  with patch.object(client.chat.completions, "create", return_value=_respuesta_groq_falsa("hola")) as mock_create:
    try:
      obtenido = preguntar_groq("\u00bfQue es un LLM?")
    except Exception as e:
      print(f"\u2717 preguntar_groq(...) lanzo {type(e).__name__}: {e}")
      return

    args, kwargs = mock_create.call_args
    mensajes = kwargs.get("messages")

    if mensajes is None:
      print("\u2717 No se estan pasando 'messages' a client.chat.completions.create")
      fallas.append("messages")
    elif len(mensajes) != 1:
      print(f"\u2717 Sin system_prompt, 'messages' debia tener 1 elemento, tiene {len(mensajes)}")
      fallas.append("largo sin system")
    elif mensajes[0] != {"role": "user", "content": "\u00bfQue es un LLM?"}:
      print(f"\u2717 El mensaje de usuario no tiene la forma esperada: {mensajes[0]}")
      fallas.append("forma mensaje user")
    else:
      print("\u2713 Sin system_prompt: 'messages' = [{'role': 'user', ...}]")

    if obtenido != "hola":
      print(f"\u2717 El valor de retorno debia ser el texto de la respuesta ('hola'), fue: {obtenido!r}")
      fallas.append("retorno")
    else:
      print("\u2713 Devuelve el texto de la respuesta (respuesta.choices[0].message.content)")

  with patch.object(client.chat.completions, "create", return_value=_respuesta_groq_falsa("hola con sistema")) as mock_create:
    preguntar_groq("\u00bfQue es un LLM?", system_prompt="Eres un asistente breve")
    args, kwargs = mock_create.call_args
    mensajes = kwargs.get("messages", [])
    if len(mensajes) == 2 and mensajes[0] == {"role": "system", "content": "Eres un asistente breve"} and mensajes[1] == {"role": "user", "content": "\u00bfQue es un LLM?"}:
      print("\u2713 Con system_prompt: 'messages' = [system, user] en ese orden")
    else:
      print(f"\u2717 Con system_prompt, 'messages' no tiene la forma esperada: {mensajes}")
      fallas.append("forma con system")

  if not fallas:
    print("\n\U0001F389 \u00a1Todo funciona correctamente!")
  else:
    print(f"\n{len(fallas)} problema(s) por corregir.")

_validar_preguntar_groq()


✓ Sin system_prompt: 'messages' = [{'role': 'user', ...}]
✓ Devuelve el texto de la respuesta (respuesta.choices[0].message.content)
✓ Con system_prompt: 'messages' = [system, user] en ese orden

🎉 ¡Todo funciona correctamente!


#### Tarea 2: Ver el efecto del system prompt

Esta celda si usa tu API key real. Corre el **mismo** prompt con y sin system prompt, y observa como cambia el comportamiento del modelo sin haber tocado ni un solo peso.


In [ ]:
pregunta = "¿Que es la temperatura en un LLM?"

print("--- Sin system prompt ---")
print(preguntar_groq(pregunta))

print("\n--- Con system prompt ---")
print(preguntar_groq(
    pregunta,
    system_prompt="Eres un profesor de ciencia de datos. Responde en maximo 2 frases, sin formulas.",
))

--- Sin system prompt ---
En los modelos de lenguaje (LLM, por sus siglas en inglés) la **temperatura** es un hiperparámetro que controla la “creatividad” o la “aleatoriedad” de las respuestas generadas. Se aplica al proceso de *sampling* (selección de la siguiente palabra) y modifica la distribución de probabilidad de los tokens que el modelo considera posibles.

---

## 1. ¿Cómo funciona la temperatura?

Cuando el modelo genera texto, primero calcula una probabilidad para cada token posible:

```
P(token) = softmax(logits)
```

La temperatura \(T\) se introduce modificando estos logits antes de aplicar el softmax:

```
P_T(token) = softmax(logits / T)
```

- **T < 1** → Los logits se amplifican (divididos por un valor menor que 1). La distribución se vuelve más “picuda”: los tokens con mayor probabilidad se vuelven aún más probables, y los de menor probabilidad se reducen casi a cero. El resultado: respuestas más **determinísticas** y coherentes, pero menos creativas.

- **T = 1** → 

## Actividad 2: Via API propia (tu propio servidor)

### Objetivo

En la Actividad 1 usamos el servidor de un tercero. Ahora vamos a montar **nuestro propio servidor** con Ollama y consultarlo de la misma forma: con peticiones HTTP.

La idea es notar que el patron cliente-servidor es identico; lo que cambia es quien es el dueno del servidor (y quien asume sus costos).


In [ ]:
# Instala Ollama, lo deja corriendo como servidor y descarga un modelo pequeno
!apt install zstd pciutils && curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # le damos tiempo al servidor para levantar

!ollama pull llama3.2:1b


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pciutils is already the newest version (1:3.7.0-6).
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 57 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



#### Tarea 1: Construir la funcion `preguntar_ollama`

Es el mismo patron de la Actividad 1, pero en vez de usar el cliente de Groq hacemos nosotros mismos la peticion HTTP con la libreria `requests`.

El servidor de Ollama expone el endpoint `http://localhost:11434/api/chat`, que recibe un JSON con `model` y `messages`, y responde con un JSON donde el texto esta en `respuesta.json()["message"]["content"]`.


In [ ]:
import requests

URL_OLLAMA = "http://localhost:11434/api/chat"
MODELO_OLLAMA = "llama3.2:1b"

def preguntar_ollama(prompt: str, system_prompt: str = None, modelo: str = MODELO_OLLAMA) -> str:
  """Envia un prompt a tu propio servidor local de Ollama y devuelve el texto de la respuesta."""

  mensajes = []

  if system_prompt:
    mensajes.append({"role": "system", "content": system_prompt})

  mensajes.append({"role": "user", "content": prompt})

  cuerpo = {
      "model": modelo,
      "messages": mensajes,
      "stream": False,
  }

  respuesta = requests.post(URL_OLLAMA, json=cuerpo)

  return respuesta.json()["message"]["content"]


**Evaluacion de implementacion**

In [ ]:
# @title
# Celda de validacion. No modificar.
from unittest.mock import patch, MagicMock

def _respuesta_ollama_falsa(texto="Respuesta simulada"):
  resp = MagicMock()
  resp.json.return_value = {"message": {"content": texto}}
  return resp

def _validar_preguntar_ollama():
  if "preguntar_ollama" not in globals():
    print("\u2717 Todavia no existe 'preguntar_ollama'. Ejecuta la celda anterior.")
    return

  fallas = []

  with patch("requests.post", return_value=_respuesta_ollama_falsa("hola")) as mock_post:
    try:
      obtenido = preguntar_ollama("\u00bfQue es un servidor?")
    except Exception as e:
      print(f"\u2717 preguntar_ollama(...) lanzo {type(e).__name__}: {e}")
      return

    if mock_post.call_args is None:
      print("\u2717 Nunca se llamo a requests.post. Falta hacer el POST a URL_OLLAMA.")
      print("\u2717 Por lo mismo no se pueden revisar el cuerpo ni el valor de retorno.")
      return

    args, kwargs = mock_post.call_args
    url = args[0] if args else kwargs.get("url")
    cuerpo = kwargs.get("json", {})

    if url != URL_OLLAMA:
      print(f"\u2717 Se esperaba que el POST fuera a {URL_OLLAMA!r}, fue a {url!r}")
      fallas.append("url")
    else:
      print("\u2713 El POST se hace a la URL correcta")

    mensajes = cuerpo.get("messages")
    if mensajes != [{"role": "user", "content": "\u00bfQue es un servidor?"}]:
      print(f"\u2717 Sin system_prompt, 'messages' debia ser [{{'role': 'user', ...}}], fue: {mensajes}")
      fallas.append("messages sin system")
    else:
      print("\u2713 Sin system_prompt, el cuerpo trae solo el mensaje de usuario")

    if obtenido != "hola":
      print(f"\u2717 El retorno debia ser 'hola' (el texto de la respuesta), fue: {obtenido!r}")
      fallas.append("retorno")
    else:
      print("\u2713 Devuelve correctamente respuesta.json()['message']['content']")

  if not fallas:
    print("\n\U0001F389 \u00a1Todo funciona correctamente!")
  else:
    print(f"\n{len(fallas)} problema(s) por corregir.")

_validar_preguntar_ollama()


✓ El POST se hace a la URL correcta
✓ Sin system_prompt, el cuerpo trae solo el mensaje de usuario
✓ Devuelve correctamente respuesta.json()['message']['content']

🎉 ¡Todo funciona correctamente!


#### Tarea 2: Comparar servidor propio vs. de terceros

Corre el mismo prompt en ambos servidores y compara **tiempo de respuesta** y **calidad**. Recuerda que el modelo de Ollama (1B) es mucho mas pequeno que el de Groq (20B).


In [ ]:
import time

prompt = "Explica en una frase que es un token en un modelo de lenguaje."

inicio = time.time()
r_groq = preguntar_groq(prompt)
t_groq = time.time() - inicio

inicio = time.time()
r_ollama = preguntar_ollama(prompt)
t_ollama = time.time() - inicio

print(f"Groq   ({t_groq:.2f}s): {r_groq}\n")
print(f"Ollama ({t_ollama:.2f}s): {r_ollama}")


Groq   (0.40s): Un token es la unidad básica de entrada (palabra, subpalabra o carácter) que el modelo de lenguaje procesa y representa mediante vectores numéricos.

Ollama (103.49s): Un token en un modelo de lenguaje es la unidad básica de un texto que puede ser un símbolo, palabra, número, signo de puntuación, etc., que se utiliza como entrada para entrenar y procesar un modelo de lenguaje.


## Actividad 3: Via libreria `transformers`

### Objetivo

Ahora cargamos un modelo **directamente en Python**, sin servidor ni peticiones HTTP: el modelo vive en la memoria de nuestro Colab (idealmente en la GPU). Esto es, a grandes rasgos, lo que hacen por dentro tanto Groq como Ollama.

Empezamos con un modelo pequeno (0.5B de parametros) que cabe comodamente en la GPU gratuita.


In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
  print("GPU:", torch.cuda.get_device_name(0))
  print(f"Memoria total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


GPU disponible: True
GPU: Tesla T4
Memoria total: 15.6 GB


#### Tarea 1: Construir la funcion `preguntar_transformers`

Hacemos a mano los tres pasos que vimos en clase.

1. **Tokenizar**: `apply_chat_template` convierte los mensajes en token IDs.
2. **Generar**: `generate` predice los tokens siguientes.
3. **Decodificar**: `decode` los convierte de vuelta a texto.

Tres detalles:

- Usa `return_dict=True`. Asi obtienes `input_ids` y `attention_mask`, que se pasan a `generate` con `**entradas`.
- `generate` devuelve el prompt **mas** la respuesta. Corta desde `entradas["input_ids"].shape[-1]`.
- Usa `clean_up_tokenization_spaces=False`: ese post-procesamiento es para tokenizers WordPiece (BERT) y dana la salida de un BPE como el de Qwen (convierte `x != y` en `x!= y`).


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from google.colab import userdata

try:
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    print("Sin HF_TOKEN: se descargaran los modelos de forma anonima (funciona para modelos publicos).")

MODELO_HF = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer_hf = AutoTokenizer.from_pretrained(MODELO_HF)
modelo_hf = AutoModelForCausalLM.from_pretrained(
    MODELO_HF,
    dtype=torch.float16,
    device_map="auto",
)


def preguntar_transformers(prompt: str, system_prompt: str = None, max_new_tokens: int = 200) -> str:
  """Genera una respuesta usando un modelo cargado localmente con transformers."""

  mensajes = []
  if system_prompt:
    mensajes.append({"role": "system", "content": system_prompt})
  mensajes.append({"role": "user", "content": prompt})

  entradas = tokenizer_hf.apply_chat_template(
      mensajes,
      add_generation_prompt=True,
      return_tensors="pt",
      return_dict=True,
  ).to(modelo_hf.device)

  salida = modelo_hf.generate(**entradas, max_new_tokens=max_new_tokens)

  return tokenizer_hf.decode(
      salida[0][entradas["input_ids"].shape[-1]:],
      skip_special_tokens=True,
      clean_up_tokenization_spaces=False,
  )

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

**Evaluacion de implementacion**

Esta validacion simula el tokenizer y el modelo, asi que no genera texto de verdad ni gasta GPU.

In [ ]:
# @title
# Celda de validacion. No modificar.
from unittest.mock import patch


class _TensorFalso:
  """Se comporta como el tensor de input_ids: 3 tokens."""
  shape = (1, 3)


class _EntradasFalsas(dict):
  """Se comporta como el BatchEncoding que devuelve apply_chat_template."""

  def __init__(self):
    super().__init__(input_ids=_TensorFalso(), attention_mask=_TensorFalso())

  def to(self, device):
    return self


class _SalidaFalsa:
  """Se comporta como el tensor de generate: 3 tokens de prompt + 2 nuevos."""

  def __getitem__(self, i):
    if i == 0:
      return [10, 11, 12, 13, 14]
    raise IndexError(i)


def _validar_preguntar_transformers():
  for nombre in ("preguntar_transformers", "tokenizer_hf", "modelo_hf"):
    if nombre not in globals():
      print(f"\u2717 Todavia no existe '{nombre}'. Ejecuta la celda anterior.")
      return

  capturado = {}

  def _apply_falso(mensajes, **kwargs):
    capturado["mensajes"] = mensajes
    capturado["add_generation_prompt"] = kwargs.get("add_generation_prompt")
    capturado["return_dict"] = kwargs.get("return_dict")
    return _EntradasFalsas()

  def _generate_falso(*args, **kwargs):
    capturado["generate_posicional"] = len(args)
    capturado["input_ids"] = kwargs.get("input_ids")
    capturado["max_new_tokens"] = kwargs.get("max_new_tokens")
    return _SalidaFalsa()

  def _decode_falso(tokens, **kwargs):
    capturado["tokens"] = list(tokens)
    capturado["skip_special_tokens"] = kwargs.get("skip_special_tokens")
    capturado["clean_up"] = kwargs.get("clean_up_tokenization_spaces")
    return "respuesta simulada"

  fallas = []

  with patch.object(tokenizer_hf, "apply_chat_template", _apply_falso), \
       patch.object(modelo_hf, "generate", _generate_falso), \
       patch.object(tokenizer_hf, "decode", _decode_falso):
    try:
      obtenido = preguntar_transformers("\u00bfQue es un token?", system_prompt="Se breve")
    except Exception as e:
      print(f"\u2717 preguntar_transformers(...) lanzo {type(e).__name__}: {e}")
      return

  esperados = [
      {"role": "system", "content": "Se breve"},
      {"role": "user", "content": "\u00bfQue es un token?"},
  ]
  if capturado.get("mensajes") == esperados:
    print("\u2713 Arma 'mensajes' con el system prompt y el mensaje de usuario, en orden")
  else:
    print(f"\u2717 'mensajes' debia ser {esperados}, fue {capturado.get('mensajes')}")
    fallas.append("mensajes")

  if capturado.get("add_generation_prompt") is True:
    print("\u2713 Usa add_generation_prompt=True al aplicar la plantilla")
  else:
    print("\u2717 Falta add_generation_prompt=True en apply_chat_template")
    fallas.append("add_generation_prompt")

  if capturado.get("return_dict") is True:
    print("\u2713 Usa return_dict=True al aplicar la plantilla")
  else:
    print("\u2717 Falta return_dict=True en apply_chat_template")
    fallas.append("return_dict")

  if capturado.get("input_ids") is not None and capturado.get("generate_posicional") == 0:
    print("\u2713 Le pasa las entradas a generate desempaquetadas (**entradas)")
  else:
    print("\u2717 generate debia recibir **entradas (no el diccionario como argumento posicional)")
    fallas.append("desempaquetado")

  if capturado.get("max_new_tokens") == 200:
    print("\u2713 Pasa max_new_tokens a generate")
  else:
    print(f"\u2717 generate debia recibir max_new_tokens=200, recibio {capturado.get('max_new_tokens')}")
    fallas.append("max_new_tokens")

  if capturado.get("tokens") == [13, 14]:
    print("\u2713 Decodifica SOLO los tokens nuevos")
  else:
    print(f"\u2717 Debia decodificar solo los tokens nuevos [13, 14], decodifico {capturado.get('tokens')}")
    print('    Recuerda cortar la salida: salida[0][entradas["input_ids"].shape[-1]:]')
    fallas.append("corte")

  if capturado.get("skip_special_tokens") is True:
    print("\u2713 Usa skip_special_tokens=True")
  else:
    print("\u2717 Falta skip_special_tokens=True en decode")
    fallas.append("skip_special_tokens")

  if capturado.get("clean_up") is False:
    print("\u2713 Usa clean_up_tokenization_spaces=False (no dana la salida del tokenizer BPE)")
  else:
    print(f"\u2717 Falta clean_up_tokenization_spaces=False en decode (es {capturado.get('clean_up')})")
    fallas.append("clean_up")

  if obtenido != "respuesta simulada":
    print(f"\u2717 Debia devolver el texto decodificado, devolvio {obtenido!r}")
    fallas.append("retorno")
  else:
    print("\u2713 Devuelve el texto decodificado")

  if not fallas:
    print("\n\U0001F389 \u00a1Todo funciona correctamente!")
  else:
    print(f"\n{len(fallas)} problema(s) por corregir.")


_validar_preguntar_transformers()

✓ Arma 'mensajes' con el system prompt y el mensaje de usuario, en orden
✓ Usa add_generation_prompt=True al aplicar la plantilla
✓ Usa return_dict=True al aplicar la plantilla
✓ Le pasa las entradas a generate desempaquetadas (**entradas)
✓ Pasa max_new_tokens a generate
✓ Decodifica SOLO los tokens nuevos
✓ Usa skip_special_tokens=True
✓ Usa clean_up_tokenization_spaces=False (no dana la salida del tokenizer BPE)
✓ Devuelve el texto decodificado

🎉 ¡Todo funciona correctamente!


#### Tarea 2: Probar el modelo y medir cuanta GPU ocupa

Aqui ya necesitamos el modelo cargado y la funcion completa (Tarea 1). Fijate en la memoria que ocupa: la vamos a comparar en la siguiente actividad.

In [ ]:
print(preguntar_transformers(
    "Explica en 3 frases que es una funcion de activacion.",
    system_prompt="Responde en espanol, de forma clara y breve.",
))

print(f"\nMemoria GPU ocupada: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


Una función de activación es un proceso donde se utiliza una variable o variables para determinar la salida de una actividad o procesamiento.

Memoria GPU ocupada: 8.84 GB


## Actividad 4: ¿Que hacer cuando el modelo no cabe en la GPU?

### Objetivo

En la Actividad 3 cargamos un modelo de 0.5B sin problema. Ahora vamos a intentar cargar uno **14 veces mas grande** (7B de parametros) en la misma GPU, ver que **falla**, entender por que, y luego lograr que funcione.

### Regla de bolsillo

Cada parametro ocupa tantos bytes como bits use su precision, dividido entre 8:

| Precision | Bytes por parametro | Memoria de un modelo de 7B |
|---|---|---|
| float32 (32 bits) | 4 | ~28 GB |
| float16 (16 bits) | 2 | ~14 GB |
| 4 bits (cuantizado) | 0.5 | ~3.5 GB |

Una GPU T4 de Colab tiene **~15 GB**, de los cuales quedan libres alrededor de 14.5 GB.


#### Tarea 1: Estimar cuanta memoria necesita un modelo

Implementa `memoria_estimada_gb(parametros_b, bits)`, que recibe el tamano del modelo en **miles de millones (B) de parametros** y la **precision en bits**, y devuelve los GB aproximados que ocupara.

$$\text{memoria en GB} \approx \text{parametros (en miles de millones)} \times \frac{\text{bits}}{8}$$

Ejemplos:

```
memoria_estimada_gb(7, 16) == 14.0     # 7B en float16
memoria_estimada_gb(7, 4)  == 3.5      # 7B cuantizado a 4 bits
memoria_estimada_gb(0.5, 16) == 1.0    # el modelo de la Actividad 3
```


In [ ]:
def memoria_estimada_gb(parametros_b: float, bits: int) -> float:
  """Estima los GB de memoria que ocupa un modelo, segun su tamano y su precision."""

  return parametros_b * (bits / 8)


**Evaluacion de implementacion**

In [ ]:
# @title
# Celda de validacion. No modificar.
from math import isclose

def _revisar_memoria(parametros_b, bits, esperado, nota=""):
  etiqueta = f"memoria_estimada_gb({parametros_b}, {bits})"
  if nota:
    etiqueta += f"   # {nota}"
  try:
    obtenido = memoria_estimada_gb(parametros_b, bits)
  except Exception as e:
    print(f"\u2717 {etiqueta}\n    lanzo {type(e).__name__}: {e}")
    return False
  if obtenido is None:
    print(f"\u2717 {etiqueta}\n    devolvio None (\u00bfte falto el return?)")
    return False
  if not isinstance(obtenido, (int, float)):
    print(f"\u2717 {etiqueta}\n    debia devolver un numero, devolvio {type(obtenido).__name__}")
    return False
  if isclose(obtenido, esperado, rel_tol=1e-6):
    print(f"\u2713 {etiqueta} = {obtenido} GB")
    return True
  print(f"\u2717 {etiqueta}\n    esperaba: {esperado}\n    obtuvo:   {obtenido}")
  return False

if "memoria_estimada_gb" not in globals():
  print("\u2717 Todavia no existe 'memoria_estimada_gb'. Ejecuta la celda anterior.")
else:
  _casos = [
      (7, 16, 14.0, "7B en float16"),
      (7, 4, 3.5, "7B cuantizado a 4 bits"),
      (0.5, 16, 1.0, "el modelo de la Actividad 3"),
      (7, 32, 28.0, "7B en float32"),
      (70, 16, 140.0, "un modelo de 70B en float16"),
  ]
  _resultados = [_revisar_memoria(*caso) for caso in _casos]
  if all(_resultados):
    print("\n\U0001F389 \u00a1Todo funciona correctamente!")
  else:
    print(f"\n{_resultados.count(False)} problema(s) por corregir.")


✓ memoria_estimada_gb(7, 16)   # 7B en float16 = 14.0 GB
✓ memoria_estimada_gb(7, 4)   # 7B cuantizado a 4 bits = 3.5 GB
✓ memoria_estimada_gb(0.5, 16)   # el modelo de la Actividad 3 = 1.0 GB
✓ memoria_estimada_gb(7, 32)   # 7B en float32 = 28.0 GB
✓ memoria_estimada_gb(70, 16)   # un modelo de 70B en float16 = 140.0 GB

🎉 ¡Todo funciona correctamente!


#### Tarea 2: Comparar lo que necesitamos contra lo que tenemos

Usa tu funcion para comparar el modelo de 7B contra la memoria libre de la GPU.


In [ ]:
import torch

libres_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
necesarios_gb = memoria_estimada_gb(7, 16)

print(f"Memoria libre en la GPU:      {libres_gb:.1f} GB")
print(f"Necesita Qwen2.5-7B (fp16):   {necesarios_gb:.1f} GB")
print(f"\n¿Cabe? {'Si' if necesarios_gb < libres_gb else 'No'}")


Memoria libre en la GPU:      14.6 GB
Necesita Qwen2.5-7B (fp16):   14.0 GB

¿Cabe? Si


#### Tarea 3: Intentarlo de todas formas (y ver el error)

Esta celda esta completa: solo ejecutala. Va a descargar el modelo (~15 GB, puede tardar varios minutos) y luego **fallara** al intentar meterlo en la GPU.

Fijate en dos cosas:

- Usamos `device_map={"": 0}` para forzar que **todo** vaya a la GPU. Si usaramos `device_map="auto"`, `accelerate` mandaria en silencio las capas que no caben a la RAM del CPU, y el modelo correria lentisimo en vez de fallar.
- El error que veras es un `torch.cuda.OutOfMemoryError`. Leelo con calma: dice cuanto intento reservar y cuanto habia disponible.


In [ ]:
from transformers import AutoModelForCausalLM

MODELO_GRANDE = "Qwen/Qwen2.5-7B-Instruct"

try:
  modelo_grande = AutoModelForCausalLM.from_pretrained(
      MODELO_GRANDE,
      dtype=torch.float16,
      device_map={"": 0},   # forzamos TODO el modelo a la GPU 0
  )
  print("El modelo cargo sin problema.")
  print("Si ves esto, tu GPU es mas grande que una T4 (revisa el tipo de entorno).")
except torch.cuda.OutOfMemoryError as e:
  print("\u2717 torch.cuda.OutOfMemoryError\n")
  print(str(e)[:600])
except Exception as e:
  print(f"\u2717 {type(e).__name__}\n")
  print(str(e)[:600])


✗ torch.cuda.OutOfMemoryError

CUDA out of memory. Tried to allocate 13.36 GiB. GPU 0 has a total capacity of 14.56 GiB of which 11.97 GiB is free. Process 18492 has 1.53 GiB memory in use. Including non-PyTorch memory, this process has 1.06 GiB memory in use. Of the allocated memory 951.45 MiB is allocated by PyTorch, and 2.55 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pyt


In [ ]:
# Limpiamos la GPU antes de volver a intentarlo
import gc

for nombre in ["modelo_grande"]:
  if nombre in globals():
    del globals()[nombre]

gc.collect()
torch.cuda.empty_cache()

print(f"Memoria GPU ocupada tras limpiar: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


Memoria GPU ocupada tras limpiar: 1.00 GB


#### Tarea 4: Hacerlo caber con cuantizacion

La **cuantizacion** guarda cada peso con menos bits (4 en vez de 16). El modelo pierde algo de precision, pero pasa de ~14 GB a ~3.5 GB y cabe sin problema.

Completa la configuracion de `BitsAndBytesConfig`:

- `load_in_4bit=True`: cargar los pesos en 4 bits.
- `bnb_4bit_compute_dtype=torch.float16`: aunque los pesos se guarden en 4 bits, los calculos se hacen en float16.
- `bnb_4bit_quant_type="nf4"`: el tipo de cuantizacion de 4 bits (NormalFloat4, el recomendado).


In [ ]:
!pip install -q bitsandbytes

from transformers import BitsAndBytesConfig

config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)


**Evaluacion de implementacion**

In [ ]:
# @title
# Celda de validacion. No modificar.

def _validar_config_4bit():
  if "config_4bit" not in globals():
    print("\u2717 Todavia no existe 'config_4bit'. Ejecuta la celda anterior.")
    return

  fallas = []

  if getattr(config_4bit, "load_in_4bit", False) is not True:
    print(f"\u2717 'load_in_4bit' debia ser True, es {getattr(config_4bit, 'load_in_4bit', None)}")
    fallas.append("load_in_4bit")
  else:
    print("\u2713 load_in_4bit = True")

  dtype = getattr(config_4bit, "bnb_4bit_compute_dtype", None)
  if dtype != torch.float16:
    print(f"\u2717 'bnb_4bit_compute_dtype' debia ser torch.float16, es {dtype}")
    fallas.append("compute_dtype")
  else:
    print("\u2713 bnb_4bit_compute_dtype = torch.float16")

  quant = getattr(config_4bit, "bnb_4bit_quant_type", None)
  if quant != "nf4":
    print(f"\u2717 'bnb_4bit_quant_type' debia ser 'nf4', es {quant!r}")
    fallas.append("quant_type")
  else:
    print("\u2713 bnb_4bit_quant_type = 'nf4'")

  if not fallas:
    print("\n\U0001F389 \u00a1Todo funciona correctamente!")
    print(f"\nCon 4 bits, el modelo de 7B deberia ocupar ~{memoria_estimada_gb(7, 4):.1f} GB en vez de {memoria_estimada_gb(7, 16):.1f} GB.")
  else:
    print(f"\n{len(fallas)} problema(s) por corregir.")

_validar_config_4bit()


✓ load_in_4bit = True
✓ bnb_4bit_compute_dtype = torch.float16
✓ bnb_4bit_quant_type = 'nf4'

🎉 ¡Todo funciona correctamente!

Con 4 bits, el modelo de 7B deberia ocupar ~3.5 GB en vez de 14.0 GB.


#### Tarea 5: Cargar el mismo modelo, ahora si

El modelo ya esta descargado (se quedo en cache en la Tarea 3), asi que esta celda solo tiene que cuantizarlo y subirlo a la GPU.


In [ ]:
from transformers import AutoTokenizer

tokenizer_grande = AutoTokenizer.from_pretrained(MODELO_GRANDE)

modelo_grande = AutoModelForCausalLM.from_pretrained(
    MODELO_GRANDE,
    quantization_config=config_4bit,
    device_map={"": 0},
)

print(f"\u2713 Modelo de 7B cargado en la GPU")
print(f"Memoria GPU ocupada: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

✓ Modelo de 7B cargado en la GPU
Memoria GPU ocupada: 14.69 GB


In [ ]:
mensajes = [
    {"role": "system", "content": "Responde en espanol, de forma breve."},
    {"role": "user", "content": "¿Que es la cuantizacion de un modelo y que se pierde al aplicarla?"},
]

entradas = tokenizer_grande.apply_chat_template(
    mensajes,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(modelo_grande.device)

salida = modelo_grande.generate(**entradas, max_new_tokens=150)

print(tokenizer_grande.decode(
    salida[0][entradas["input_ids"].shape[-1]:],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
))

La cuantización de un modelo se refiere a la reducción de la precisión de los pesos del modelo, generalmente al redondearlos a enteros más pequeños o a potencias de dos. Se pierde precisión en las predicciones del modelo debido a esta simplificación.


## Actividad 5: Configurando parametros del modelo

### Objetivo

Extender `preguntar_groq` (Actividad 1) con los parametros `top_p`, `max_tokens` y `stop`, para controlar como genera el texto.

Recuerda de la clase:

- `temperature`: aplana o agudiza la distribucion (0 = siempre el token mas probable).
- `top_p`: se queda con los tokens que suman `p` de probabilidad.
- `max_tokens`: corta por longitud.
- `stop`: corta al aparecer cierto texto.


In [ ]:
client = Groq(api_key=userdata.get("GROQ_API_KEY"))

def preguntar_groq_configurable(
    prompt: str,
    system_prompt: str = None,
    modelo: str = MODELO_GROQ,
    temperature: float = 0.7,
    top_p: float = 1.0,
    max_tokens: int = 256,
    stop: list = [],
) -> str:
  """Igual que preguntar_groq, pero permite configurar top_p, max_tokens y stop."""

  mensajes = []
  if system_prompt:
    mensajes.append({"role": "system", "content": system_prompt})
  mensajes.append({"role": "user", "content": prompt})

  respuesta = client.chat.completions.create(
      model=modelo,
      messages=mensajes,
      temperature=temperature,
      top_p=top_p,
      max_completion_tokens=max_tokens,
      stop=stop,
  )

  return respuesta.choices[0].message.content


**Evaluacion de implementacion**

In [ ]:
# @title
# Celda de validacion. No modificar.
from unittest.mock import patch, MagicMock

def _respuesta_configurable_falsa(texto="ok"):
  resp = MagicMock()
  resp.choices = [MagicMock()]
  resp.choices[0].message.content = texto
  return resp

def _validar_preguntar_groq_configurable():
  if "preguntar_groq_configurable" not in globals():
    print("\u2717 Todavia no existe 'preguntar_groq_configurable'. Ejecuta la celda anterior.")
    return

  fallas = []
  with patch.object(client.chat.completions, "create", return_value=_respuesta_configurable_falsa()) as mock_create:
    preguntar_groq_configurable("hola", temperature=0.3, top_p=0.9, max_tokens=50, stop=["FIN"])
    args, kwargs = mock_create.call_args

    for nombre, esperado in [("temperature", 0.3), ("top_p", 0.9), ("max_completion_tokens", 50), ("stop", ["FIN"])]:
      if kwargs.get(nombre) != esperado:
        print(f"\u2717 Se esperaba {nombre}={esperado!r}, se envio {kwargs.get(nombre)!r}")
        fallas.append(nombre)
      else:
        print(f"\u2713 Se envia {nombre}={esperado!r} correctamente")

  if not fallas:
    print("\n\U0001F389 \u00a1Todo funciona correctamente!")
  else:
    print(f"\n{len(fallas)} problema(s) por corregir.")

_validar_preguntar_groq_configurable()


✓ Se envia temperature=0.3 correctamente
✓ Se envia top_p=0.9 correctamente
✓ Se envia max_completion_tokens=50 correctamente
✓ Se envia stop=['FIN'] correctamente

🎉 ¡Todo funciona correctamente!


#### Experimentar con la temperatura

Corre el mismo prompt con distintas temperaturas y observa cuanto cambia la respuesta.


In [ ]:

prompt_creativo = "Inventa un nombre para un modelo de lenguaje entrenado con recetas de cocina."

for temperatura in [0, 0.5, 1.0, 1.5]:
  respuesta = preguntar_groq_configurable(prompt_creativo, temperature=temperatura, max_tokens=None)
  print(f"\ntemperature={temperatura}: \n{respuesta}")



temperature=0: 
**Algunas ideas de nombres para tu modelo de lenguaje especializado en recetas de cocina:**

| # | Nombre | Comentario breve |
|---|--------|------------------|
| 1 | **CocinaGPT** | Directo y reconocible, combina “cocina” con la familiaridad de GPT. |
| 2 | **SaborAI** | Evoca el sentido del gusto y la inteligencia artificial. |
| 3 | **RecetAI** | Juego de palabras entre “receta” y “AI”. |
| 4 | **ChefBot** | Sugiere un asistente culinario profesional. |
| 5 | **GastroLang** | Refleja la combinación de gastronomía y lenguaje. |
| 6 | **SazónNet** | “Sazón” transmite sabor y “Net” sugiere una red neuronal. |
| 7 | **TastyTalk** | En inglés, suena amigable y apetitoso. |
| 8 | **FlavorForge** | Implica la creación de sabores con tecnología. |
| 9 | **CulinAI** | Fusiona “culinaria” con “AI”. |
|10 | **PantryPal** | Un compañero de cocina que sabe de todo. |

Puedes elegir el que más resuene con la identidad que quieras darle al modelo o combinar elementos de varios par

#### Experimentar con `stop`

`stop` corta la generacion apenas aparece cierto texto. Corre la celda y observa donde se detiene.


In [ ]:
system_prompt_lista = "Enumera 5 tipos de transformers. Usa el formato '1.', '2.', etc."

print("--- Sin stop ---")
print(preguntar_groq_configurable(system_prompt_lista, temperature=0, max_tokens=None))

print("\n--- Con stop=['3.'] ---")
print(preguntar_groq_configurable(system_prompt_lista, temperature=0, max_tokens=None, stop=["3."]))


--- Sin stop ---
1. **BERT (Bidirectional Encoder Representations from Transformers)** – Modelo de encoder bidireccional que se entrena con tareas de “masking” y “next sentence prediction” para capturar contexto en ambas direcciones.  
2. **GPT (Generative Pre‑trained Transformer)** – Modelo de decoder autoregresivo que se entrena con lenguaje generativo, ideal para tareas de generación de texto y completado de frases.  
3. **T5 (Text‑to‑Text Transfer Transformer)** – Convierte todas las tareas de NLP en un problema de “texto a texto”, entrenando el modelo para generar una salida de texto a partir de una entrada de texto.  
4. **RoBERTa (Robustly Optimized BERT Approach)** – Variante de BERT que optimiza el pre‑entrenamiento eliminando la tarea de “next sentence prediction” y usando más datos y mayor longitud de secuencia.  
5. **DistilBERT** – Versión más ligera y rápida de BERT, obtenida mediante distilación de conocimiento, manteniendo un rendimiento cercano al original con menos pa

### Preguntas para discutir

**1. ¿Cuando usarias cada una de las 3 formas de correr un LLM (API de terceros, servidor propio, libreria directa)? ¿Que ganas y que pierdes con cada una?**

- **API de terceros**: cuando quieres empezar ya y no tienes infraestructura. Ganas modelos grandes sin comprar GPUs y sin mantenimiento; pierdes control, mandas tus datos a un externo, pagas por token y dependes de que no deprecien el modelo (nos paso con Llama en este mismo notebook).
- **Servidor propio (Ollama, vLLM)**: cuando los datos no pueden salir de la organizacion, el volumen es alto, o necesitas latencia estable. Ganas privacidad y costo fijo; pierdes tiempo en montarlo y quedas limitado a modelos que quepan en tu hardware.
- **Libreria directa (`transformers`)**: cuando necesitas tocar el modelo por dentro: cuantizar, hacer fine-tuning, leer los logits, medir memoria. Es lo que hicimos en las Actividades 3 y 4. No sirve para producir a escala, porque cada peticion se procesa sin batching ni las optimizaciones de un servidor real.

**2. En la Tarea 3 de la Actividad 4 forzamos `device_map={"": 0}` para que el error apareciera. Si hubieramos usado `device_map="auto"`, el modelo habria cargado pero mucho mas lento. ¿Cual de los dos comportamientos preferirias en produccion?**

En general, **fallar rapido**. Un error al arrancar se detecta en el despliegue; un modelo que carga y responde 20 veces mas lento se detecta cuando los usuarios se quejan. La degradacion silenciosa es peor que la caida ruidosa porque nadie sabe que hay un problema.

La excepcion es cuando la alternativa a "lento" es "nada": un proceso por lotes que corre de noche puede permitirse el offload a CPU con tal de terminar. La clave es que sea una decision explicita y monitoreada, no un accidente.

**3. La cuantizacion a 4 bits redujo la memoria a la cuarta parte. ¿Que crees que se pierde? ¿Como lo medirias?**

Se pierde **precision en los pesos**: cada numero se guarda con menos bits, asi que el modelo queda ligeramente distinto del original. En la practica el dano suele ser pequeno pero no nulo, y se nota mas en tareas que exigen exactitud (matematicas, codigo, razonamiento largo) que en escribir texto fluido.

Como medirlo: no basta con leer las dos respuestas y opinar. Se necesita un conjunto de casos con respuesta conocida (por ejemplo preguntas de opcion multiple o problemas con resultado verificable), correr las dos versiones sobre los mismos casos y comparar el porcentaje de aciertos. Tambien conviene medir velocidad, porque la cuantizacion no siempre acelera.

**4. Si `temperature=0`, ¿deberia la respuesta ser siempre exactamente igual? ¿Por que podria no serlo?**

En teoria si: con `temperature=0` siempre se escoge el token mas probable, sin azar.

En la practica casi nunca lo es. Las razones son varias: las sumas en punto flotante en GPU no son asociativas y el resultado depende de como se agrupen las operaciones, que a su vez depende de con cuantas peticiones te toque compartir el lote; si dos tokens quedan empatados, cualquier diferencia minuscula inclina la decision hacia otro lado y a partir de ahi el texto diverge; y ademas el proveedor puede cambiar de hardware, de version o de modelo sin avisarte.

La leccion practica: `temperature=0` reduce muchisimo la variabilidad, pero **no** es una garantia de reproducibilidad. Si necesitas resultados reproducibles, guarda las salidas en vez de asumir que las puedes regenerar identicas.

### Extensiones opcionales

- **Streaming**: usa `stream=True` en `client.chat.completions.create` y observa como llegan los tokens uno por uno.
- **Comparar calidad tras cuantizar**: corre el mismo prompt en el modelo de 7B cuantizado y en el de 0.5B de la Actividad 3. ¿Cual responde mejor?
- **8 bits**: repite la Actividad 4 con `load_in_8bit=True`. ¿Cabe en la T4? ¿Cuanta memoria ocupa?
- **Function calling**: revisa la documentacion de Groq sobre *tool use* y arma una funcion que el modelo pueda "llamar".